# FIFA World Cup Predictor

This notebook demonstrates the complete pipeline:
1. Load data (World Cup matches + international results)
2. Engineer features (ELO ratings, rolling stats, head-to-head)
3. Train XGBoost classifier
4. Predict match outcomes
5. Simulate 2026 World Cup tournament
6. Generate reports and visualizations

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Reload modules to pick up code changes
import importlib
import sys
if 'predictor' in sys.modules:
    import predictor.config
    import predictor.data_loader
    import predictor.feature_engineer
    import predictor.model_trainer
    import predictor.predictor_api
    import predictor.simulator
    import predictor.reporter
    importlib.reload(predictor.config)
    importlib.reload(predictor.data_loader)
    importlib.reload(predictor.feature_engineer)
    importlib.reload(predictor.model_trainer)
    importlib.reload(predictor.predictor_api)
    importlib.reload(predictor.simulator)
    importlib.reload(predictor.reporter)

from predictor.config import DEFAULT_MODEL_PATH, DEFAULT_OUTPUT_DIR, RANDOM_SEED, FEATURE_COLS, OUTCOME_INT_MAP
from predictor.data_loader import DataLoader
from predictor.feature_engineer import FeatureEngineer
from predictor.model_trainer import ModelTrainer
from predictor.predictor_api import PredictorAPI
from predictor.simulator import TournamentSimulator
from predictor.reporter import ReportGenerator

DATA_DIR = '.'
CUTOFF_YEAR = 2006
N_RUNS = 1000
print('Imports OK')

## Step 1: Load Data

In [ ]:
loader = DataLoader(DATA_DIR)
matches_df, players_df, cups_df, results_df = loader.load()
print(f'WC Matches:            {len(matches_df):>6} rows')
print(f'International results: {len(results_df):>6} rows  ({results_df["year"].min()}-{results_df["year"].max()})')
print(f'Players:               {len(players_df):>6} rows')
print(f'\nYear column info:')
print(f'  Type: {matches_df["Year"].dtype}')
print(f'  NaN count: {matches_df["Year"].isna().sum()}')
print(f'  Min: {matches_df["Year"].min()}, Max: {matches_df["Year"].max()}')
matches_df.head()

## Step 2: Feature Engineering

In [ ]:
print('Building features (may take 1-2 minutes)...')
fe = FeatureEngineer(matches_df, players_df, results_df)
features_df = fe.build_features()
print(f'Feature matrix: {features_df.shape}')
print(f'Outcome distribution:')
print(features_df['outcome'].value_counts().to_string())
features_df.head()

## Step 3: Train Model

In [ ]:
trainer = ModelTrainer(features_df, cutoff_year=CUTOFF_YEAR, model_path=DEFAULT_MODEL_PATH)
result = trainer.train()
trainer.save()
print('Test set metrics:')
for k, v in result.metrics.items():
    print(f'  {k}: {v:.4f}')
print(f'\nTop 10 features by importance:')
print(result.feature_importance.nlargest(10).to_string())

## Step 4: Test Predictions

In [ ]:
api = PredictorAPI(result.model, fe)

test_pairs = [
    ('Brazil', 'Germany'),
    ('France', 'Argentina'),
    ('Spain', 'England'),
]
print('Sample match predictions:')
for home, away in test_pairs:
    try:
        pred = api.predict(home, away)
        print(f'{home:15} vs {away:15}  |  Home: {pred.home_win_prob:.3f}  Draw: {pred.draw_prob:.3f}  Away: {pred.away_win_prob:.3f}  =>  {pred.predicted_label}')
    except ValueError as e:
        print(f'  Skipping: {e}')

## Step 5: Setup 2026 World Cup Groups

In [ ]:
WC_2026_GROUPS = {
    'A': ['Brazil', 'Argentina', 'Colombia', 'Ecuador'],
    'B': ['France', 'England', 'Spain', 'Portugal'],
    'C': ['Germany', 'Netherlands', 'Belgium', 'Denmark'],
    'D': ['Italy', 'Croatia', 'Serbia', 'Switzerland'],
    'E': ['Uruguay', 'Chile', 'Mexico', 'USA'],
    'F': ['Morocco', 'Senegal', 'Nigeria', 'Cameroon'],
    'G': ['Japan', 'South Korea', 'Australia', 'Iran'],
    'H': ['Saudi Arabia', 'Tunisia', 'Algeria', 'Poland'],
}

known = set(api._team_lookup.keys())
valid_groups = {}
for g, teams in WC_2026_GROUPS.items():
    valid = [t for t in teams if t.lower() in known]
    if len(valid) >= 2:
        valid_groups[g] = valid

valid_teams = [t for teams in valid_groups.values() for t in teams]
print(f'Valid groups: {len(valid_groups)}, teams: {len(valid_teams)}')
for g, t in valid_groups.items():
    print(f'  Group {g}: {t}')

## Step 6: Simulate Tournament & Generate Reports

In [ ]:
if len(valid_groups) >= 8:
    simulator = TournamentSimulator(api, n_runs=N_RUNS, seed=RANDOM_SEED)
    sim_result = simulator.simulate(valid_teams, valid_groups)

    reporter = ReportGenerator(DEFAULT_OUTPUT_DIR)
    reporter.bar_chart(sim_result)
    reporter.feature_importance(result.feature_importance)
    reporter.summary_csv(sim_result)

    # Confusion matrix on test set
    test_df = features_df[features_df['year'] >= CUTOFF_YEAR]
    y_true = test_df['outcome'].tolist()
    y_pred_int = result.model.predict(test_df[list(FEATURE_COLS)])
    y_pred = [OUTCOME_INT_MAP[i] for i in y_pred_int]
    reporter.confusion_matrix(y_true, y_pred)

    print('\nTop 10 predicted 2026 WC winners:')
    for i, team in enumerate(sim_result.ranked_teams[:10], 1):
        prob = sim_result.win_probabilities[team]
        sf_prob = sim_result.semifinal_probabilities[team]
        print(f'  {i:2d}. {team:<20} win={prob:.3f}  semi={sf_prob:.3f}')
else:
    print(f'Only {len(valid_groups)} valid groups — need 8. Check team name mapping.')